# Stage 00a.1 — Dataset Overview

Bird's-eye view of the dataset's health and size at every pipeline stage.

| Cell | What it does |
|------|--------------|
| 1 | Load config & paths |
| 2 | **Stage 1 — Raw inventory**: folders, images, disk size |
| 3 | **Stage 2 — Post-triage**: kept vs discarded by SigLIP |
| 4 | **Stage 3 — Labelling CSVs**: crops mapped, labels assigned, human review queue |
| 5 | **Delta report**: combined funnel table + reduction metrics |

All cells are **read-only** — nothing is modified on disk.

In [1]:
# ── Cell 1: Config & paths ────────────────────────────────────────────────────
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir      = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config

cfg = load_config()

RAW_DIR    = Path(cfg["paths"]["raw_images"])
TRIAGE_DIR = Path(cfg["paths"]["triage"])
DATA_DIR   = Path(cfg["paths"]["data"])

print("Paths resolved:")
print(f"  raw_images : {RAW_DIR}")
print(f"  triage     : {TRIAGE_DIR}")
print(f"  data       : {DATA_DIR}")
print(f"  raw exists : {RAW_DIR.exists()}")
print(f"  triage exists: {TRIAGE_DIR.exists()}")
print(f"  data exists  : {DATA_DIR.exists()}")

Paths resolved:
  raw_images : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
  triage     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/triage
  data       : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data
  raw exists : True
  triage exists: True
  data exists  : True


## Stage 1 — Raw Images

In [2]:
# ── Cell 2: Raw inventory ─────────────────────────────────────────────────────
import os
import pandas as pd

if not RAW_DIR.exists():
    print(f"✗ raw_images dir not found: {RAW_DIR}")
    raise SystemExit

patent_folders = sorted(p for p in RAW_DIR.iterdir() if p.is_dir())

rows = []
total_bytes = 0
for folder in patent_folders:
    pngs = list(folder.glob("*.png"))
    folder_bytes = sum(f.stat().st_size for f in pngs)
    total_bytes += folder_bytes
    rows.append({
        "patent_id":   folder.name,
        "raw_images":  len(pngs),
        "size_mb":     round(folder_bytes / 1e6, 3),
    })

df_raw = pd.DataFrame(rows)

n_folders  = len(df_raw)
n_raw_imgs = int(df_raw["raw_images"].sum())
raw_gb     = total_bytes / 1e9

print(f"Patent folders  : {n_folders:,}")
print(f"Total raw images: {n_raw_imgs:,}")
print(f"Disk size (raw) : {raw_gb:.2f} GB")
print()
print("Top 10 folders by image count:")
display(df_raw.nlargest(10, "raw_images").reset_index(drop=True))

Patent folders  : 1,614
Total raw images: 13,835
Disk size (raw) : 0.87 GB

Top 10 folders by image count:


,patent_id,raw_images,size_mb
0,CA3096221A1_81206876,41,2.531
1,KR20260072583A_84020260072583,41,2.531
2,EP4467450A1_87196346,13,1.711
3,EP4563464A1_93741590,13,0.731
4,AU2020100605A4_70776151,12,1.089
5,CA2967402A1_58709316,12,1.201
6,CN101885295A_43071443,12,0.426
7,CN103786881A_50663046,12,0.138
8,CN106005373A_57096574,12,0.425
9,CN107521686A_60748639,12,0.289


## Stage 2 — Post-Triage (SigLIP filter)

In [3]:
# ── Cell 3: Triage summary ────────────────────────────────────────────────────
import json

if not TRIAGE_DIR.exists():
    print(f"✗ triage dir not found — run 00a2_triage_filter first.")
    raise SystemExit

summary_csv = TRIAGE_DIR / "triage_summary.csv"

# Prefer the pre-built summary CSV; fall back to reading all JSONs.
if summary_csv.exists():
    df_triage = pd.read_csv(summary_csv)
    print(f"Loaded triage_summary.csv  ({len(df_triage)} patents)")
else:
    print("triage_summary.csv not found — rebuilding from per-patent JSONs...")
    triage_rows = []
    for jp in sorted(TRIAGE_DIR.glob("*.json")):
        if jp.stem == "triage_summary":
            continue
        with open(jp) as fh:
            d = json.load(fh)
        triage_rows.append({
            "patent_id":     d["patent_id"],
            "total_images":  d["total"],
            "flagged_count": d["flagged"],
            "flagged_ratio": round(d["flagged"] / d["total"], 4) if d["total"] else 0.0,
        })
    df_triage = pd.DataFrame(triage_rows)
    print(f"Rebuilt from {len(df_triage)} JSON files.")

n_triaged_patents = len(df_triage)
n_triage_total    = int(df_triage["total_images"].sum())
n_discarded       = int(df_triage["flagged_count"].sum())
n_kept_triage     = n_triage_total - n_discarded
n_untriaged       = n_folders - n_triaged_patents   # raw folders not yet scored

# Also count size of kept images directly from triage JSONs
kept_bytes = 0
for jp in sorted(TRIAGE_DIR.glob("*.json")):
    if jp.stem == "triage_summary":
        continue
    with open(jp) as fh:
        d = json.load(fh)
    patent_dir = RAW_DIR / d["patent_id"]
    for fig in d["figures"]:
        if fig["keep"]:
            p = patent_dir / fig["file"]
            try:
                kept_bytes += p.stat().st_size
            except FileNotFoundError:
                pass

kept_gb = kept_bytes / 1e9

print(f"\nPatents triaged       : {n_triaged_patents:,}  (of {n_folders:,} raw folders)")
print(f"Images scored         : {n_triage_total:,}")
print(f"Images KEPT           : {n_kept_triage:,}")
print(f"Images DISCARDED      : {n_discarded:,}")
print(f"Discard rate          : {100*n_discarded/max(1,n_triage_total):.1f}%")
print(f"Disk size (kept imgs) : {kept_gb:.2f} GB")
if n_untriaged:
    print(f"\n⚠  {n_untriaged} raw folder(s) not yet triaged")

print("\nTop 10 patents by discarded count:")
display(
    df_triage.nlargest(10, "flagged_count")
    [["patent_id", "total_images", "flagged_count", "flagged_ratio"]]
    .reset_index(drop=True)
)

Loaded triage_summary.csv  (1614 patents)

Patents triaged       : 1,614  (of 1,614 raw folders)
Images scored         : 13,835
Images KEPT           : 10,417
Images DISCARDED      : 3,418
Discard rate          : 24.7%
Disk size (kept imgs) : 0.69 GB

Top 10 patents by discarded count:


,patent_id,total_images,flagged_count,flagged_ratio
0,CN116280179A_86786318,12,12,1.0000
1,CN119099860A_93717638,12,12,1.0000
2,CN120191516A_96063272,12,12,1.0000
3,US2021103298A1_75274827,12,12,1.0000
4,US2022048639A1_71127804,12,12,1.0000
5,US2022055743A1_71126277,12,12,1.0000
6,US2022194556A1_72608819,12,12,1.0000
7,US2024317395A1_92803854,12,12,1.0000
8,CN119239921A_94035552,12,11,0.9167
9,CN120482345A_96675626,12,11,0.9167


## Stage 3 — Labelling CSVs

In [4]:
# ── Cell 4: Labelling CSV audit ───────────────────────────────────────────────
# Reads data/ CSVs produced by downstream labelling stages.
# Gracefully skips any CSV that doesn't exist yet.

def _safe_read(path: Path, desc: str) -> pd.DataFrame | None:
    if not path.exists():
        print(f"  ✗ {desc}: not found ({path.name})")
        return None
    df = pd.read_csv(path)
    print(f"  ✓ {desc}: {len(df):,} rows  ({path.name})")
    return df

print("Data directory CSVs:")
df_crops   = _safe_read(DATA_DIR / "crops_mapping.csv",   "crops_mapping")
df_review  = _safe_read(DATA_DIR / "needs_human_review.csv", "needs_human_review")
df_desc    = _safe_read(DATA_DIR / "descriptions.csv",    "descriptions")
df_epo     = _safe_read(DATA_DIR / "epo_coverage_report.csv", "epo_coverage")

# ── Crops summary ─────────────────────────────────────────────────────────────
n_crops        = len(df_crops) if df_crops is not None else 0
n_labeled      = 0
n_needs_review = len(df_review) if df_review is not None else 0

if df_crops is not None and "label" in df_crops.columns:
    n_labeled = int(df_crops["label"].notna().sum())
    n_unlabeled = n_crops - n_labeled
    print(f"\nCrops mapped          : {n_crops:,}")
    print(f"  Labels assigned     : {n_labeled:,}")
    print(f"  Unlabeled (no label): {n_unlabeled:,}")
    print(f"  Needs human review  : {n_needs_review:,}")

    if "method" in df_crops.columns:
        print("\nLabel method breakdown:")
        display(df_crops["method"].value_counts().rename_axis("method").reset_index(name="count"))

# ── Patents with descriptions ─────────────────────────────────────────────────
n_with_desc = len(df_desc) if df_desc is not None else 0
if n_with_desc:
    print(f"\nPatents with descriptions: {n_with_desc:,}")

Data directory CSVs:
  ✓ crops_mapping: 221 rows  (crops_mapping.csv)
  ✓ needs_human_review: 13 rows  (needs_human_review.csv)
  ✓ descriptions: 1,639 rows  (descriptions.csv)
  ✗ epo_coverage: not found (epo_coverage_report.csv)

Crops mapped          : 221
  Labels assigned     : 208
  Unlabeled (no label): 13
  Needs human review  : 13

Label method breakdown:


,method,count
0,doclayout_easyocr,206
1,doclayout_qwen,13
2,doclayout_qwen_strip,2



Patents with descriptions: 1,639


## Delta Report — Pipeline Funnel

In [5]:
# ── Cell 5: Funnel summary table ──────────────────────────────────────────────
import math

stages = [
    {
        "stage":       "Stage 1 — Raw",
        "patents":     n_folders,
        "images":      n_raw_imgs,
        "size_gb":     round(raw_gb, 2),
        "delta_imgs":  0,
        "pct_of_raw":  100.0,
    },
    {
        "stage":       "Stage 2 — Post-Triage (kept)",
        "patents":     n_triaged_patents,
        "images":      n_kept_triage,
        "size_gb":     round(kept_gb, 2),
        "delta_imgs":  -(n_discarded),
        "pct_of_raw":  round(100 * n_kept_triage / max(1, n_raw_imgs), 1),
    },
    {
        "stage":       "Stage 3 — Crops mapped",
        "patents":     None,
        "images":      n_crops if n_crops else None,
        "size_gb":     None,
        "delta_imgs":  None,
        "pct_of_raw":  round(100 * n_crops / max(1, n_raw_imgs), 1) if n_crops else None,
    },
    {
        "stage":       "Stage 3 — Labels assigned",
        "patents":     None,
        "images":      n_labeled if n_labeled else None,
        "size_gb":     None,
        "delta_imgs":  None,
        "pct_of_raw":  round(100 * n_labeled / max(1, n_raw_imgs), 1) if n_labeled else None,
    },
]

df_funnel = pd.DataFrame(stages)
df_funnel = df_funnel.where(df_funnel.notna(), other="—")

print("═" * 72)
print(" DATASET FUNNEL REPORT")
print("═" * 72)
display(df_funnel.set_index("stage"))

print()
print(f"Total images removed by triage : {n_discarded:,}  "
      f"({100*n_discarded/max(1,n_raw_imgs):.1f}% of raw)")
if n_raw_imgs > 0:
    print(f"Active dataset after triage    : {n_kept_triage:,} images  "
          f"({100*n_kept_triage/n_raw_imgs:.1f}% of raw)")
if raw_gb > 0 and kept_gb > 0:
    saved_gb = raw_gb - kept_gb
    print(f"Storage saved by triage        : {saved_gb:.2f} GB  "
          f"({100*saved_gb/raw_gb:.1f}% of raw size)")
if n_needs_review:
    print(f"Images pending human review    : {n_needs_review:,}")

════════════════════════════════════════════════════════════════════════
 DATASET FUNNEL REPORT
════════════════════════════════════════════════════════════════════════


,patents,images,size_gb,delta_imgs,pct_of_raw
stage,,,,,
Stage 1 — Raw,1614.0,13835,0.87,0.0,100.0
Stage 2 — Post-Triage (kept),1614.0,10417,0.69,-3418.0,75.3
Stage 3 — Crops mapped,—,221,—,—,1.6
Stage 3 — Labels assigned,—,208,—,—,1.5



Total images removed by triage : 3,418  (24.7% of raw)
Active dataset after triage    : 10,417 images  (75.3% of raw)
Storage saved by triage        : 0.18 GB  (20.9% of raw size)
Images pending human review    : 13
